# gem

> Simple utilities for working with Google's Gemini API

This notebook provides a minimal interface to Google's Gemini API. The goal is to make it dead simple to:

1. Generate text with just a prompt
2. Analyze files (PDFs, images, **MP4 videos**) 
3. Process videos (YouTube URLs or **local MP4 files**)

All through a single `gem()` function that just works.

In [74]:
#| default_exp gem

In [75]:
#| hide
from nbdev.showdoc import *

## Setup

First, make sure you have your Gemini API key set:

In [76]:
#| export
import asyncio, mimetypes, os, time, threading
from pathlib import Path
from fastcore.all import *
from google import genai
from google.genai import types
from fastprogress import progress_bar

In [77]:
#| hide
try:
    import nest_asyncio
    nest_asyncio.apply()
except ModuleNotFoundError:
    pass


In [78]:
# export GEMINI_API_KEY='your-api-key'
assert os.environ.get("GEMINI_API_KEY"), "Please set GEMINI_API_KEY environment variable"

## Building blocks

Let's start with the simple helper functions that make everything work.

### Client creation

We need a Gemini client to talk to the API:

In [79]:
#|export
async def _to_thread(func, /, *args, **kwargs):
    "Run blocking helper on a background thread"
    return await asyncio.to_thread(func, *args, **kwargs)


In [80]:
#|export
def _client():
    "Get Gemini client context manager"
    return genai.Client()

In [81]:
#|hide
async def _test_to_thread():
    res = await _to_thread(lambda x: x + 1, 1)
    assert res == 2

await _test_to_thread()


In [82]:
#|hide
c = _client()
assert c is not None
assert hasattr(c, 'models')

## Video upload

In [83]:
#|export
def upload_file(pth):
    if not Path(pth).exists(): raise ValueError(f"File {pth} does not exist.")
    with _client() as c:
        f = c.files.upload(file=pth)
        time.sleep(2)
        for i in progress_bar(range(30)):
            try:
                f = c.files.get(name=f.name)
                if f.state == 'ACTIVE': return f
                elif f.state == 'FAILED': raise Exception(f'File processing for {pth} failed.')
                time.sleep(10)
            except: pass # because the gemini file thing is jank
        raise Exception(f'Timeout processing {pth}')

In [84]:
#| export
async def upload_file_async(pth):
    "Async wrapper around upload_file"
    return await _to_thread(upload_file, pth)


In [85]:
#| hide
from unittest.mock import patch

async def _test_upload_file_async():
    with patch('__main__.upload_file', return_value='ok') as mock_upload:
        result = await upload_file_async('foo')
    mock_upload.assert_called_once_with('foo')
    assert result == 'ok'

await _test_upload_file_async()


In [86]:
myfile = upload_file("_videos/test_video.mp4")
assert myfile.state == 'ACTIVE'

### Converting attachments to Parts

Gemini expects different types of content (files, URLs) to be wrapped in "Parts". This helper handles that conversion:

In [87]:
#| export
_VIDEO_EXTS = {'.mp4', '.mpeg', '.mov', '.avi', '.flv', '.mpg', '.webm', '.wmv', '.3gpp'}
_AUDIO_EXTS = {'.wav', '.mp3', '.aiff', '.aac', '.ogg', '.flac', '.m4a'}
_INLINE_MIME_MAP = {
    '.pdf': 'application/pdf',
    '.png': 'image/png',
    '.jpg': 'image/jpeg',
    '.jpeg': 'image/jpeg',
    '.gif': 'image/gif',
    '.txt': 'text/plain',
    '.vtt': 'text/plain',
    '.md': 'text/markdown',
    '.json': 'text/plain',  # Gemini doesn't accept application/json
    '.yaml': 'text/yaml',
    '.yml': 'text/yaml',
    '.toml': 'text/plain',
    '.ipynb': 'text/plain'
}

def _normalize_attachments(o):
    "Ensure attachments are always a list"
    if isinstance(o, list): return o
    return [o] if o else []

def _is_media_file(p: Path):
    return p.suffix.lower() in (_VIDEO_EXTS | _AUDIO_EXTS)

def _mime_for_path(p: Path):
    mime = _INLINE_MIME_MAP.get(p.suffix.lower())
    if mime: return mime
    guessed_mime, _ = mimetypes.guess_type(str(p))
    if guessed_mime is None:
        raise ValueError(f"Cannot determine MIME type for file: {p}. Unsupported extension: {p.suffix}")
    return guessed_mime


In [88]:
#| hide
assert _normalize_attachments(None) == []
assert _normalize_attachments('foo') == ['foo']
assert _normalize_attachments(['foo', 'bar']) == ['foo', 'bar']
assert _is_media_file(Path('_videos/test_video.mp4')) is True
assert _is_media_file(Path('_test_files/sample.txt')) is False
assert _mime_for_path(Path('_test_files/sample.txt')) == 'text/plain'


In [89]:
#| export
def _is_url(s):
    "Check if string is a URL"
    if not isinstance(s, str): return False
    return (s.startswith('http://') or 
            s.startswith('https://') or 
            s.startswith('www.') or 
            'youtube.com' in s or 
            'youtu.be' in s)

def _make_part(o):
    "Convert object to Gemini Part (sync)"
    if isinstance(o, types.File):
        return types.Part.from_uri(file_uri=o.uri, mime_type=o.mime_type)
    if isinstance(o, (str, Path)):
        p = Path(o)
        if p.exists():
            if _is_media_file(p):
                f = upload_file(o)
                return types.Part.from_uri(file_uri=f.uri, mime_type=f.mime_type)
            return types.Part.from_bytes(mime_type=_mime_for_path(p), data=p.read_bytes())
        elif _is_url(o):
            return types.Part.from_uri(file_uri=o, mime_type='video/*')
        else:
            raise ValueError(f"Could not parse file or url: {o}")
    return None

In [90]:
#| export
async def _make_part_async(o):
    "Async wrapper around _make_part"
    return await _to_thread(_make_part, o)


In [91]:
#| hide
async def _test_make_part_async():
    part = await _make_part_async('_test_files/sample.txt')
    assert part.inline_data.mime_type == 'text/plain'

await _test_make_part_async()


In [92]:
_part = _make_part('_videos/test_video.mp4')
_part

Part(
  file_data=FileData(
    file_uri='https://generativelanguage.googleapis.com/v1beta/files/9as0r29gc45g',
    mime_type='video/mp4'
  )
)

In [93]:
#|hide
# Test text file formats
txt_part = _make_part('_test_files/sample.txt')
assert txt_part.inline_data.mime_type == 'text/plain'

vtt_part = _make_part('_test_files/sample.vtt')
assert vtt_part.inline_data.mime_type == 'text/plain'

md_part = _make_part('_test_files/sample.md')
assert md_part.inline_data.mime_type == 'text/markdown'

## The main interface

Now we can build our main `gem()` function that handles all use cases:

In [94]:
#| export
def _parts_from_attachments(prompt, attachments):
    "Build Parts list from normalized attachments"
    parts = [types.Part.from_text(text=prompt)]
    for attachment in attachments:
        if part := _make_part(attachment):
            parts.insert(0, part)
    return parts

#| export
async def _parts_from_attachments_async(prompt, attachments):
    "Async Parts builder"
    parts = [types.Part.from_text(text=prompt)]
    for attachment in attachments:
        part = await _make_part_async(attachment)
        if part:
            parts.insert(0, part)
    return parts


In [95]:
#| hide
async def _test_parts_builders():
    attachments = _normalize_attachments('_test_files/sample.txt')
    sync_parts = _parts_from_attachments("prompt", attachments)
    async_parts = await _parts_from_attachments_async("prompt", attachments)
    assert len(sync_parts) == len(async_parts) == 2
    assert sync_parts[0].inline_data.mime_type == 'text/plain'
    assert async_parts[0].inline_data.mime_type == 'text/plain'

await _test_parts_builders()


In [96]:
#| export
def _content_payload(prompt, attachments, parts):
    "Return prompt or Content depending on attachments"
    return types.Content(role='user', parts=parts) if attachments else prompt


In [97]:
#| hide
parts = _parts_from_attachments("hello", _normalize_attachments('_test_files/sample.txt'))
content = _content_payload("hello", ['_test_files/sample.txt'], parts)
assert isinstance(content, types.Content)
assert content.parts[-1].text == 'hello'
assert _content_payload("hello", [], parts) == "hello"


In [98]:
#| export
def _build_config(thinking, search, parts):
    config_dict = {
        'thinking_config': types.ThinkingConfig(thinking_budget=thinking),
        'response_mime_type': 'text/plain'
    }
    if any(getattr(p, 'file_data', None) and getattr(p.file_data, 'mime_type', '').startswith('video')
           for p in parts):
        config_dict['media_resolution'] = 'MEDIA_RESOLUTION_LOW'
    tools = []
    if search:
        tools.append(types.Tool(google_search=types.GoogleSearch()))
    if tools:
        config_dict['tools'] = tools
    return types.GenerateContentConfig(**config_dict)


In [99]:
#| hide
parts = _parts_from_attachments("prompt", _normalize_attachments('_test_files/sample.txt'))
cfg = _build_config(thinking=-1, search=True, parts=parts)
assert cfg.response_mime_type == 'text/plain'
assert cfg.thinking_config.thinking_budget == -1
assert len(cfg.tools) == 1


In [100]:
#| export
def _chunk_text(chunk):
    "Best-effort way to pull text out of a streaming chunk"
    if hasattr(chunk, 'text') and chunk.text:
        return chunk.text
    cand = getattr(chunk, 'candidates', None)
    if cand:
        candidate = cand[0]
        content = getattr(candidate, 'content', None)
        if content and getattr(content, 'parts', None):
            part = content.parts[-1]
            if hasattr(part, 'text') and part.text:
                return part.text
    return str(chunk)

#| export
async def _stream_generate_content(model, contents, cfg):
    loop = asyncio.get_running_loop()
    queue = asyncio.Queue()

    def _run_stream():
        try:
            with _client() as client:
                for chunk in client.models.generate_content_stream(model=model, contents=contents, config=cfg):
                    fut = asyncio.run_coroutine_threadsafe(queue.put(chunk), loop)
                    fut.result()
        except Exception as e:
            asyncio.run_coroutine_threadsafe(queue.put(e), loop).result()
        finally:
            asyncio.run_coroutine_threadsafe(queue.put(None), loop).result()

    threading.Thread(target=_run_stream, daemon=True).start()

    while True:
        item = await queue.get()
        if item is None:
            break
        if isinstance(item, Exception):
            raise item
        yield _chunk_text(item)



In [101]:
#| export
def _generate_content(model, contents, cfg):
    with _client() as client:
        return client.models.generate_content(model=model, contents=contents, config=cfg)


In [102]:
#| export
def gem(prompt, # Text prompt
        o=None, # Optional file/URL attachment or list of attachments
        model='gemini-2.5-flash',
        thinking=-1,
        search=False):
    "Generate content with Gemini"
    attachments = _normalize_attachments(o)
    parts = _parts_from_attachments(prompt, attachments)
    contents = _content_payload(prompt, attachments, parts)
    cfg = _build_config(thinking, search, parts)
    resp = _generate_content(model, contents, cfg)
    return resp.text

In [103]:
#| export
async def gem_async(prompt,
                    o=None,
                    model='gemini-2.5-flash',
                    thinking=-1,
                    search=False,
                    stream=False):
    "Async wrapper around gem using background threads. Set stream=True for an async iterator."
    attachments = _normalize_attachments(o)
    parts = await _parts_from_attachments_async(prompt, attachments)
    contents = _content_payload(prompt, attachments, parts)
    cfg = _build_config(thinking, search, parts)
    if stream:
        return _stream_generate_content(model, contents, cfg)
    resp = await _to_thread(_generate_content, model, contents, cfg)
    return resp.text


In [104]:
#| hide
async def _test_gem_async_stream():
    stream = await gem_async("Summarize this markdown file.", "_test_files/sample.md", stream=True)
    chunks = []
    async for chunk in stream:
        if chunk:
            chunks.append(chunk)
    assert ''.join(chunks).strip()

await _test_gem_async_stream()



In [105]:
await gem_async(
    "Summarize this markdown file in one sentence.",
    "_test_files/sample.md"
)


'This markdown file is a test document demonstrating basic formatting and evaluating text/markdown MIME type support.'

### Async interface

Need non-blocking calls? Use `gem_async` (plus helpers like `upload_file_async`) to run the same flow via `asyncio.to_thread`, so it plays nicely inside notebooks and event-loop frameworks.


#### Streaming (async only)

Pass `stream=True` to `gem_async` to get an async iterator. Await the call once to obtain the stream, then iterate over it to consume chunks as Gemini sends them.


In [110]:
stream = await gem_async(
    "Write a detailed synopsis followed by a bullet summary of this video.",
    "https://youtu.be/1x3k0V2IITo",
    stream=True
)
async for chunk in stream:
    print(chunk, end="")



The video features Antoine Chaffin, a Research Engineer at LightOn, discussing the limitations of single vector search and introducing multi-vector models, also known as late interaction models, as a superior alternative, especially for modern RAG (Retrieval-Augmented Generation) pipelines.

Chaffin begins by introducing himself and his background, which includes a PhD in multimodal misinformation detection, where he studied information retrieval and generative models. At LightOn, he focuses on information retrieval, particularly encoder models and late interaction, co-creating the ModernBERT encoder and the PyLate library. He also works on OCR-free RAG pipelines and visual rerankers.

He then dives into the core topic, explaining dense (single) vector search. This method involves feeding a query and documents into a transformer model (like ModernBERT) to create contextualized vector representations for each token. These token vectors are then "pooled" into a single vector (using max, 

## Examples

One function handles everything:
- Just text? Pass a prompt.
- Have a file? Pass it as the second argument.
- Got a YouTube URL? Same thing.

Let's test it out:

## Text generation

The simplest case - just generate some text:

In [111]:
gem("Write a haiku about Python programming")

'Simple, readable code,\nIndented, clean, logic flows,\nPower in each line.'

In [112]:
await gem_async("Write a haiku about Python programming")

'Clear, simple code,\nLike a serpent, logic flows,\nIdeas come to life.'

## Video analysis

Perfect for creating YouTube chapters or summaries:

In [113]:
prompt = "5 word summary of this video."
gem(prompt, "https://youtu.be/1x3k0V2IITo")

'This video explains that traditional single vector search (dense models) compress token information, leading to limitations in out-of-domain and long-context scenarios. The speaker introduces "late interaction" (multi-vector) models as a solution, which keep all token vectors and use MaxSim for similarity, providing better generalization and interpretability. They highlight the PyLate library for easier implementation of these models, noting improved performance in reasoning-intensive and long-context retrieval tasks compared to dense models.\n\nHere\'s a 5-word summary: **Multi-vector search outperforms single-vector.**'

In [114]:
await gem_async(prompt, "https://youtu.be/1x3k0V2IITo")

'The speaker, Antoine Chaffin, explains the limitations of single vector search in information retrieval due to pooling operations that compress token information. He then introduces "late interaction" (multi-vector) models as a solution, which avoid pooling and use a token-level similarity operation (MaxSim) to retain all information. This approach improves performance, especially for out-of-domain, long-context, and reasoning-intensive retrieval tasks, even outperforming larger single-vector models. He also highlights PyLate, a library for training and evaluating these multi-vector models, and discusses future research avenues like reducing storage costs and applying late interaction to other modalities.'

### Local MP4 Video Analysis

You can also analyze local MP4 video files:

In [115]:
# Example with local MP4 file (if you have one)
gem("Summarize this video in 3 sentences.", "_videos/test_video.mp4")

'A bald man with glasses introduces the video as a "super short test recording." He then proceeds to recite the numbers "1 2 3 4 5 6." The man concludes the test by stating his name as "Hamil Hussain."'

In [116]:
await gem_async("Summarize this video in 3 sentences.", "_videos/test_video.mp4")

'The video features a man with a bald head and glasses conducting a short test recording. During the test, he recites the numbers one through six. He concludes the recording by stating his name as Hamil Hussein.'

### File analysis

Great for extracting information from PDFs or images:

In [117]:
gem("3 sentence summary of this presentation.", "NewFrontiersInIR.pdf")

'This presentation explores new frontiers in Information Retrieval (IR) by enabling models to follow complex instructions and perform reasoning, akin to large language models (LLMs). It introduces two main models: "Promptriever," a fast bi-encoder trained with synthetic instructions for promptable retrieval, and "Rank1," a powerful but slower cross-encoder that utilizes test-time compute for deeper reasoning. These instruction-trained retrievers significantly enhance search capabilities by unlocking new types of natural language queries, moving beyond simple keywords, and achieving higher accuracy in nuanced retrieval tasks.'

In [118]:
await gem_async("3 sentence summary of this presentation.", "NewFrontiersInIR.pdf")

'This presentation explores "New Frontiers in IR: Instruction Following and Reasoning," arguing that traditional search, even with LLM wrappers, hasn\'t fully evolved to handle complex user instructions. It introduces two main models: "Promptriever," an instruction-trained bi-encoder that can be prompted like a language model for efficient instruction following, and "Rank1," a cross-encoder that uses test-time compute for robust, reasoning-based reranking. Both models demonstrate significant improvements over existing methods across various tasks, unlocking new types of natural language queries and enhancing the ability to retrieve nuanced and highly relevant documents.'

In [119]:
gem("What's in this image?", "anton.png")

'This image is a promotional thumbnail, likely for a video or article, focusing on a technical topic, most probably related to AI/Machine Learning, specifically "vectors" and "RAG" (Retrieval-Augmented Generation).\n\nHere\'s a breakdown of the elements:\n\n*   **Background:** A dark, solid blue-black color, providing a high contrast for the text and graphics.\n*   **Person:** In the bottom left, a young man with light skin and brown hair is visible from the chest up, smiling broadly and looking slightly to his right (viewer\'s left). He is wearing a white V-neck t-shirt.\n*   **Emoji:** Directly above the man\'s head, slightly to the left, is a yellow "sad face" or "worried face" emoji, creating a visual contrast with his smile.\n*   **Text:**\n    *   In the top left, in large, bold white letters: "Single Vector?"\n    *   Below that, overlapping with the man\'s head and the emoji, in large, bold yellow letters: "YOU\'RE MISSING OUT"\n*   **Graphics (Network/Diagram):**\n    *   **To

In [120]:
await gem_async("What's in this image?", "anton.png")

'This image is a digital graphic, likely a thumbnail for a video or article, set against a dark blue or black background.\n\nHere\'s a breakdown of its contents:\n\n1.  **Text:**\n    *   At the top left, in large white font, is the phrase: "Single Vector?"\n    *   Below that, in large yellow font, are the stacked phrases: "YOU\'RE MISSING OUT".\n\n2.  **Emoji & Person:**\n    *   Above the word "YOU\'RE," there is a yellow sad or worried face emoji.\n    *   To the left and slightly below the text and emoji, a young man with brown hair and a white V-neck shirt is smiling broadly, looking directly at the viewer.\n\n3.  **Diagram/Graphic:**\n    *   On the right side of the image, there is a glowing blue, abstract diagram resembling a network or graph. It consists of multiple interconnected blue circles (nodes) and lines (edges).\n    *   An arrow from a cluster of these nodes points downwards to a rectangular box.\n    *   This box is dark blue with rounded corners and contains the wh

### Text file analysis

Works with common text formats like .txt, .vtt, .md:

In [121]:
gem("What type of file is this?", "_test_files/sample.txt")


'This is a **plain text file**.\n\nIt even mentions its MIME type within the content: "text/plain".'

In [122]:

await gem_async("What type of file is this?", "_test_files/sample.txt")

'This is a **plain text file**.\n\nMore specifically, it aligns with the `text/plain` MIME type, as it explicitly states and its content is simple human-readable characters without any special formatting, encoding, or binary data.'

In [123]:
gem("How many subtitle entries are in this file?", "_test_files/sample.vtt")

'There are **2** subtitle entries in this VTT file.'

In [124]:
gem("List the items in this markdown file.", "_test_files/sample.md")

'Based on the markdown file provided, the items listed are:\n\n1.  Item 1\n2.  Item 2'

### Change Model

You can also control the model and thinking time:

In [125]:
gem("What is Hamel Husain's current job?", model="gemini-2.5-pro")

"Based on his public profiles, Hamel Husain's current job is **Head of Data Science & Machine Learning at Argonaut**.\n\nHe is also well-known for his previous role as a Principal Machine Learning Scientist at **GitHub**, where he created popular open-source tools like `nbdev`, `fastpages`, and `ghapi`."

### Grounded Search

As you can see, grounded search is required to get things right sometimes!

In [126]:
gem("What is Hamel Husain's current job?.", search=True)

'Hamel Husain is currently working as an independent consultant, assisting companies in building and operationalizing AI products, with a particular focus on Large Language Models (LLMs). He also co-teaches a course titled "AI Evals for Engineers & PMs."\n\nPreviously, Hamel Husain held the position of Staff Machine Learning Engineer at GitHub. He has over 20 years of experience as a machine learning engineer, having worked with companies like Airbnb and GitHub, where his work included early LLM research utilized by OpenAI for code understanding. He has also contributed to numerous popular open-source machine learning tools.'

In [127]:
await gem_async("What is Hamel Husain's current job?.", search=True)

'Hamel Husain is currently an independent consultant, assisting companies with building and operationalizing AI products, especially those involving Large Language Models (LLMs). He also works as an AI consultant at Parlance Labs and co-teaches a course titled "AI Evals for Engineers & PMs".\n\nPreviously, Hamel Husain held the position of Staff Machine Learning Engineer at GitHub. He has over 20 years of experience as a machine learning engineer and has contributed to numerous popular open-source machine learning tools.'

### Multiple Attachments

You can analyze multiple files/URLs at once by passing a list:

In [128]:
prompt = "Is this PDF and YouTube video related or are they different talks? Answer with very short yes/no answer."
gem(prompt, ["https://youtu.be/Trps2swgeOg?si=yK7CO0Zk4E1rfp6s", "NewFrontiersInIR.pdf"])

'No'

In [129]:
await gem_async(prompt, ["https://youtu.be/YB3b-wPbSH8?si=WI0LqflY5SYIsRz9", "NewFrontiersInIR.pdf"])

'Yes.'

In [130]:
gem("What do these slides and this video have in common in terms of content/subject matter if at all? Provide a 1 sentence summary of each.", ["NewFrontiersInIR.pdf", "_videos/test_video.mp4"])

'The video is a brief test recording where a man introduces himself and recites numbers to check audio quality.\n\nThe slides present research on "New Frontiers in IR," exploring how to make information retrieval systems understand and follow natural language instructions and perform reasoning, much like large language models, using methods like "Promptriever" and "Rank1."\n\nThese two pieces of content have **no commonality** in terms of subject matter or content. The video is a personal audio test, while the slides are a technical presentation on advanced information retrieval.'

In [131]:
await gem_async("What do these slides and this video have in common in terms of content/subject matter if at all? Provide a 1 sentence summary of each.", ["NewFrontiersInIR.pdf", "_videos/test_video.mp4"])


'The video and the slides have **no commonality** in terms of content or subject matter.\n\n*   **Video Summary:** The video shows a man conducting a brief audio test, reciting numbers and his name.\n*   **Slideshow Summary:** The slideshow presents "Promptriever" and "Rank1" as novel information retrieval models capable of instruction following and complex reasoning, similar to large language models.'

## Export -

In [73]:
#| hide
import nbdev; nbdev.nbdev_export()